# Step 3: Build the Baseline Anomaly Detector

**Why this step:** Before touching anything deep-learning related, we need a simple, well-understood point of comparison. If our later, more complex VAE-based approach cannot beat this baseline, that is itself an important and honest thing to report, not a failure to hide.

This notebook extracts simple statistical features from each cached light curve, fits an Isolation Forest across the full dataset to get a per-light-curve anomaly score, and sanity-checks the results by plotting the highest and lowest scoring light curves.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import skew, kurtosis
from sklearn.ensemble import IsolationForest
import matplotlib.pyplot as plt

## Config

In [ ]:
CACHE_DIR = Path("lightcurve_cache")
# Where Step 2 saved all the cleaned, resampled .npy light curve files.

N_TOP_PLOTS = 5
# How many highest-scoring (most anomalous) and lowest-scoring (most
# "normal") light curves to plot side by side for a visual sanity check.

## Step A: Extract simple statistical features from one light curve

In [ ]:
def extract_features(flux, gap_mask):
    """
    Compute a handful of simple statistical descriptors of a light curve's
    flux array. These are cheap, well-understood, and require no
    learning -- exactly what a baseline should be built from.
    """

    # WHY we filter with gap_mask here: gap_mask==False points are
    # interpolation guesses from Step 2, not real data (they sit inside
    # large observation gaps). Including them would bias every statistic
    # below with fake, artificially flat/smooth values that don't reflect
    # anything the star actually did.
    valid_flux = flux[gap_mask]

    # If cleaning + gap-masking left too few usable points, computing
    # statistics (especially skew/kurtosis, which need a reasonable sample
    # size to be meaningful) isn't reliable -- signal this with None so
    # the caller can skip this light curve entirely.
    if len(valid_flux) < 10:
        return None

    features = {
        "std": np.std(valid_flux),
        # Standard deviation: overall spread of the flux values around
        # their mean. A star with big brightness swings has high std; a
        # very steady star has low std.

        "ptp": np.ptp(valid_flux),
        # Peak-to-peak range = max(flux) - min(flux). Captures the single
        # most extreme excursion in the whole light curve.

        "roughness": np.mean(np.abs(np.diff(valid_flux))),
        # Point-to-point roughness: average absolute difference between
        # CONSECUTIVE points. Smooth curve -> low roughness; jittery
        # curve -> high roughness.

        "p5": np.percentile(valid_flux, 5),
        "p95": np.percentile(valid_flux, 95),
        # 5th and 95th percentiles: robust alternatives to min/max,
        # less sensitive to one single freak data point.

        "skewness": skew(valid_flux),
        # Skewness measures ASYMMETRY of the flux distribution.
        #   - Negative skew: occasional deep DIPS (transit-like events).
        #   - Positive skew: occasional bright FLARES.
        #   - Near zero: roughly symmetric variation.

        "kurtosis": kurtosis(valid_flux),
        # Kurtosis measures how "heavy-tailed" the distribution is --
        # high kurtosis means mostly steady with occasional sharp
        # spikes/dips standing out.
    }

    return features

## Step B: Load every cached light curve and build a feature table

In [ ]:
def build_feature_dataframe():
    """
    Loop over every cached .npy file in CACHE_DIR, extract features from
    each, and assemble them into one DataFrame -- one row per (star,
    sector) light curve.
    """
    rows = []

    # CACHE_DIR.glob("*.npy") finds every .npy file in the cache folder --
    # this naturally picks up every light curve Step 2 successfully
    # processed and saved, regardless of which target/sector it came from.
    for cache_file in CACHE_DIR.glob("*.npy"):

        data = np.load(cache_file, allow_pickle=True).item()

        features = extract_features(data["flux"], data["gap_mask"])

        if features is None:
            # extract_features returned None -- too few valid points in
            # this particular light curve. Log it and move on rather than
            # crashing or silently including a garbage row.
            print(f"  Skipping {cache_file.name} (too few valid points)")
            continue

        # Attach identifying metadata so we can trace any flagged anomaly
        # back to its actual star, sector, and cached file later.
        features["tic_id"] = data["tic_id"]
        features["sector"] = data["sector"]
        features["cache_file"] = cache_file.name

        rows.append(features)

    # Convert our list of per-light-curve feature dictionaries into one
    # big DataFrame -- pandas automatically turns each dict into a row,
    # matching dictionary keys to column names.
    return pd.DataFrame(rows)

## Step C: Fit Isolation Forest across the full feature table

In [ ]:
def fit_isolation_forest(feature_df, feature_columns):
    """
    Fit an IsolationForest on the numeric feature columns and return a
    per-row anomaly score (higher = more anomalous).
    """

    # Pull out just the numeric feature columns as a plain numpy array --
    # this is the "X" matrix IsolationForest expects: one row per light
    # curve, one column per feature (std, ptp, roughness, etc.).
    X = feature_df[feature_columns].values

    iso_forest = IsolationForest(
        n_estimators=200,
        # Number of random trees to build. More trees -> more stable,
        # reliable average path-length estimates. 200 is a reasonable
        # middle ground between stability and speed.

        contamination="auto",
        # Lets the model decide its own internal threshold for what
        # fraction of points to treat as outliers, rather than guessing
        # a fixed percentage upfront.

        random_state=42,
        # Fixes the randomness so re-running this notebook gives
        # identical results every time -- important for reproducibility.
    )

    iso_forest.fit(X)

    # score_samples returns the model's internal anomaly score, where
    # LOWER (more negative) values mean MORE anomalous.
    raw_scores = iso_forest.score_samples(X)

    # Flip the sign so HIGHER anomaly_score = MORE anomalous -- more
    # intuitive to sort/filter/interpret downstream.
    anomaly_score = -raw_scores

    return anomaly_score, iso_forest

## Step D: Sanity-check by plotting highest and lowest scoring light curves

In [ ]:
def plot_light_curve(cache_file, ax, title):
    """Load one cached light curve and plot it on a given matplotlib axis."""
    data = np.load(CACHE_DIR / cache_file, allow_pickle=True).item()
    ax.plot(data["time"], data["flux"], linewidth=0.8)
    ax.set_title(title, fontsize=9)
    ax.set_xlabel("Time")
    ax.set_ylabel("Flux")

In [ ]:
def sanity_check_plots(feature_df, n=N_TOP_PLOTS):
    """
    Plot the N highest-scoring (most anomalous) and N lowest-scoring
    (most "normal") light curves side by side.

    WHY THIS STEP MATTERS: a numeric anomaly score alone doesn't tell us
    whether the Isolation Forest is actually catching something visually
    unusual, or just reacting to noise in how we computed the features.
    The highest-scoring curves should visibly LOOK different/unusual
    compared to the lowest-scoring ones.
    """

    # nlargest/nsmallest pick out the N rows with the highest/lowest
    # values in the "anomaly_score" column.
    top_anomalies = feature_df.nlargest(n, "anomaly_score")
    most_normal = feature_df.nsmallest(n, "anomaly_score")

    # Set up a 2-row grid of plots: top row = anomalies, bottom row = normal.
    fig, axes = plt.subplots(2, n, figsize=(4 * n, 6))

    for i, (_, row) in enumerate(top_anomalies.iterrows()):
        title = f"TIC {row['tic_id']} S{row['sector']}\nscore={row['anomaly_score']:.2f}"
        plot_light_curve(row["cache_file"], axes[0, i], title)

    for i, (_, row) in enumerate(most_normal.iterrows()):
        title = f"TIC {row['tic_id']} S{row['sector']}\nscore={row['anomaly_score']:.2f}"
        plot_light_curve(row["cache_file"], axes[1, i], title)

    axes[0, 0].set_ylabel("MOST ANOMALOUS", fontsize=11, fontweight="bold")
    axes[1, 0].set_ylabel("MOST NORMAL", fontsize=11, fontweight="bold")

    plt.tight_layout()
    plt.show()

## Run

In [ ]:
# Build the feature table from every cached light curve.
feature_df = build_feature_dataframe()
print(f"Extracted features for {len(feature_df)} light curves")

# Quick sanity check -- verify the table actually has the columns we expect
# before trying to train on it.
print(feature_df.columns.tolist())
print(feature_df.shape)

In [ ]:
# The columns IsolationForest will actually be trained on -- everything
# except the identifying/metadata columns (tic_id, sector, cache_file).
feature_columns = ["std", "ptp", "roughness", "p5", "p95", "skewness", "kurtosis"]

anomaly_score, iso_forest = fit_isolation_forest(feature_df, feature_columns)
feature_df["anomaly_score"] = anomaly_score

# Sort so the most anomalous light curves are at the top.
feature_df = feature_df.sort_values("anomaly_score", ascending=False)

print("\nTop 10 most anomalous light curves:")
print(feature_df[["tic_id", "sector", "anomaly_score"]].head(10))

In [ ]:
# Save the full results to disk -- this becomes our REFERENCE POINT for
# comparing against the VAE-based approach in Step 4.
feature_df.to_csv("baseline_anomaly_scores.csv", index=False)
print("Saved -> baseline_anomaly_scores.csv")

In [ ]:
# Visual sanity check -- do the highest-scoring curves actually LOOK
# unusual compared to the lowest-scoring ones?
sanity_check_plots(feature_df, n=N_TOP_PLOTS)